# Bag of Tokens (LeetCode #948)

## Problem Description

You have an initial **power** of `power`, an initial **score** of `0`, and a bag of `tokens` where `tokens[i]` is the value of the `i-th` token (0-indexed).

Your goal is to maximize your total **score** by potentially playing each token in one of two ways:

- **Face-up**: If your current power is at least `tokens[i]`, you may play the `i-th` token face up, losing `tokens[i]` power and gaining `1` score.
- **Face-down**: If your current score is at least `1`, you may play the `i-th` token face down, gaining `tokens[i]` power and losing `1` score.

Each token may be played **at most once** and in **any order**. You do not have to play all the tokens.

Return the **largest possible score** you can achieve after playing any number of tokens.

### Example 1:
```
Input: tokens = [100], power = 50
Output: 0
Explanation: Playing the only token is not profitable since you can't play it face up (power < 100) and you have no score to play it face down.
```

### Example 2:
```
Input: tokens = [200,100], power = 150
Output: 1
Explanation: Play token1 (100) face up, power becomes 50, score becomes 1. No more profitable moves.
```

### Example 3:
```
Input: tokens = [100,200,300,400], power = 200
Output: 2
Explanation: Play tokens in this order:
- Play token0 (100) face up, power=100, score=1
- Play token3 (400) face down, power=500, score=0
- Play token1 (200) face up, power=300, score=1
- Play token2 (300) face up, power=0, score=2
```

### Constraints:
- `0 <= tokens.length <= 1000`
- `0 <= tokens[i], power < 10^4`

## Approach 1: Brute Force (Recursive)

**Intuition:** Try all possible combinations of playing tokens face-up or face-down and find the maximum score.

**Algorithm:**
1. For each token, try three options: skip it, play face-up, or play face-down
2. Recursively solve for remaining tokens
3. Track maximum score achieved

In [ ]:
# Brute Force Solution (Recursive - TLE for large inputs)
class Solution:
    def bagOfTokensScore(self, tokens: list[int], power: int) -> int:
        self.max_score = 0
        
        def backtrack(used, power, score):
            self.max_score = max(self.max_score, score)
            
            for i in range(len(tokens)):
                if used[i]:
                    continue
                
                used[i] = True
                
                # Try face up
                if power >= tokens[i]:
                    backtrack(used, power - tokens[i], score + 1)
                
                # Try face down
                if score >= 1:
                    backtrack(used, power + tokens[i], score - 1)
                
                used[i] = False
        
        backtrack([False] * len(tokens), power, 0)
        return self.max_score

**Complexity Analysis (Brute Force):**
- Time: O(n! × 2^n) - Exploring all permutations with face-up/down choices
- Space: O(n) - Recursion stack depth

---
## Approach 2: Optimal (Greedy + Two Pointers)

**Intuition:** Sort the tokens. To maximize score:
- Play the **smallest** tokens face-up (spend least power for +1 score)
- Play the **largest** tokens face-down (gain most power for -1 score)

This greedy approach works because we want to trade power for score using cheap tokens, and trade score for power using expensive tokens.

**Algorithm:**
1. Sort tokens in ascending order
2. Use two pointers: left (smallest) and right (largest)
3. If we can afford the smallest token, play it face-up (gain score)
4. Else if we have score and tokens left, play largest face-down (gain power)
5. Else, we can't make any more moves
6. Track maximum score throughout

In [ ]:
# Optimal Solution - Greedy + Two Pointers
class Solution:
    def bagOfTokensScore(self, tokens: list[int], power: int) -> int:
        tokens.sort()
        left, right = 0, len(tokens) - 1
        score = 0
        max_score = 0
        
        while left <= right:
            if power >= tokens[left]:
                # Play smallest token face-up: spend power, gain score
                power -= tokens[left]
                score += 1
                left += 1
                max_score = max(max_score, score)
            elif score > 0:
                # Play largest token face-down: gain power, lose score
                power += tokens[right]
                score -= 1
                right -= 1
            else:
                # Can't do anything
                break
        
        return max_score

**Complexity Analysis (Optimal):**
- Time: O(n log n) - Sorting dominates; two-pointer scan is O(n)
- Space: O(1) - Only constant extra space (excluding sort)

## Summary

| Approach | Time | Space |
|----------|------|-------|
| Brute Force (Recursive) | O(n! × 2^n) | O(n) |
| Optimal (Greedy + Two Pointers) | O(n log n) | O(1) |

**Key Insight:** Sort and greedily spend power on cheapest tokens (face-up) and recover power from most expensive tokens (face-down). Always track max score since trading score for power may temporarily decrease it.